<a href="https://colab.research.google.com/github/davidmkidd/UK-Supermarket-Carbon-Emissions/blob/main/UKSmktComp_Emissions_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://evoviz.uk/wp-content/uploads/2026/04/Food_Divider_trans2.png">

# Supermarket Emissions Data

When you first encounter a dataset, you should explore its size, structure, and field definitions and content, with the aim of understanding the dataset and determining editing and processing steps required to clean, summarise, and reformat data to meet its use purpose.

In this tutorial you will undertake EDA, data cleaning, and preprocessing on the Emissions and Energy data extracted from the Annual Reports of eleven large UK supermarkets to create a dataset with a single record for each retailer, KPI and year.

# Set-up

### Libraries and Data

In [ ]:
# Load libraries
library(dplyr)   # Data manipulation
library(ggplot2)
library(data.table) # Like statement

# Download 'raw' UKSmktComp emissions and energy data from GitHub
url <- "https://raw.githubusercontent.com/davidmkidd/UK-Supermarket-Carbon-Emissions/refs/heads/main/retailer_emissions.csv"
download_path <- "/content/retailer_emissions.csv"
download.file(url, destfile = download_path, mode = "wb")
carbon <- read.csv("/content/retailer_emissions.csv", header=TRUE, stringsAsFactors=FALSE)

# Download retailer data
url <- "https://raw.githubusercontent.com/davidmkidd/UK-Supermarket-Carbon-Emissions/refs/heads/main/retailer_data.csv"
download_path <- "/content/retailer_data.csv"
download.file(url, destfile = download_path, mode = "wb")
retailer.data <- read.csv("/content/retailer_data.csv", header=TRUE, stringsAsFactors=FALSE)

# Data Structure and Cleaning

UKSmktComp is supplied as a single 'flat' table containing all unique KPI values reported (original and recalculations) by retailers.

## Emissions Data

*retailer_emissions.csv* contains emssions and energy metrics extracted from the 2013 - 2025 annual financial reports of eleven of the UKs largest supermarket retailers (Table 1).

<br>

<figcaption>Table 1. retailer_emissions.csv fields</figcaption>

| Field | Type | Description |
|----------|----------|----------|
| Retailer_Id | Text | Retailer identifier |
| Source_Year | Integer | Year of source report |
| KPI | Text | Name of KPI, ‘Scope 1’, ‘Scope 2’, ‘Scope 1 and 2’, ‘Scope 3’, ‘Scope 1 – 3’, or ‘Energy’ |
| Method | Text | ‘Location’, ‘Market’ or NA|
| KPI_Type | Text | ‘Measure’ or ‘Intensity’ |
| Year | Integer | Year of KPI value |
| Source | Text | Record source |
| Source_Value |Numeric | Value in report |
| Source_Unit | Text | Unit in report |
| Value | Numeric | Source_Value standardized to tCO2e or GWh |
| Unit | Text | Standardized unit |
| Unit_Neum | Unit numerator of standardisted intensity |
| Unit_Denom_1 | Text | 1st part of standardized intensity unit denominator, ‘m2’ or ‘£m’ |
| Unit_Denom_2 | Text | 2nd part of standardized intensity unit denominator, ‘Sales’, , ‘Sales Revenue’, ‘Turnover’, ‘Sales Floor’, ‘GIA’, ‘Stores and DC’. |
| Business_Scope | Text | Business extent of the value which may be organisational or geographical |

<br>

### Notes:

1. Metrics

> KPIs are either **absolute metrics** calculated by applying a *method* to a *business scope* or **intensity metrics** scaled to a measure of retailer 'size'. ***kpi_type*** identifies values as 'Measure' or 'Intensity'.

> Metrics for retailer and year are defined by:

> * ***retailer*** / ***retailer_code*** define the retailer
> * ***year*** is the value year.
> * ***kpi*** is the name of the KPI.
> * ***unit*** is the unit of measurement, further divided into, *unit_neum*, *unit_denom_1* and *unit_denom_2*.
> * ***business_scope*** is the geographical or organisational boundary for which values are calculated.

3. Accounting Period.

> Reports vary in accounting period with most covering either the previous year beginning January (calendar year), February, or April (financial year). Reports without an explicit accounting period are assumed to be the same as other reported years for that company. Values are assigned to the year the reports cover the majority of.

4. Multiple Values

> Systematic recalculation following significant restructuing and ad-hoc corrections result in multiple values for the same metric and year. All unique metric values are recorded with values repeated in more than one report associated to the annual report in which it is first published- its *source_year* .

5. Standardization

> KPIs may be reported in different units, e.g. tCO2e or 1000's, tCO2eKWh or GWh. 'Raw reported' values and units are recorded in *Source_Value* and *Source_Unit*, with strandardised values in *Value* field and units in *Unit*, *Unit_Neum*, *Unit_Denom_1*, and *Unit_Denom_2*. Emissions are standardised to 'tCO2e' and energy to 'GWh'. Area-based intensity metrics are standardised to 'm2' and monetory intensity to million pounds '£m'.

4. Business Scope

> Values are recorded for different business scopes, for example 'UK' or 'Global' operations, or the entire UK business or just the suprmarket component. Values for all business scopes are recorded. Business scope may change between reports.

6. Intensity

> Reported intensity values vary in magnitude within and between retailer kpis. Values within retatiler kpi time series that are one or more orders of magnitude different from the majority of values in the time series are 'corrected' by multiplying so that values are equivalent. Similarly, Sainsburys reported Scope 1 Intensity by area are multipled by 100 to align with the values of other retailers. Value changes are recorded in the 'Note' field.










## Fields and Records

Let's explore the store table structure and content.


In [ ]:
# names() returns field names as a vector
names(carbon)

In [ ]:
# nrow() returns the number of records
nrow(carbon)


Variable values should be explored individally and in relation to one-another to identify pattern. Here, values are explored using desciptive statistics and anomalous values cleaned for purpose.

Use `table()` to explore the number of values in categorical fields.

#### Retailers

In [ ]:
table(carbon$retailer_code, useNA = "always")

The number of retailer records depends on:

* How long the retailer has been reporting.

* The number of KPIs reported by retailers.

* The number of times values have been recalculated.

#### KPIs

In [ ]:
table(carbon$kpi, useNA = "always")

KPIs:

* 'Scope 1 and 2' is the most frequently reported category.

* 'Scope 1 and 2' may be reported seperately and/or combined.

* 'Scope 3' and 'Energy' are least often reported.

In [ ]:
table(carbon$kpi_type, useNA = "always")

KPI metric type:

* There are four times more absolute quantities than intensity metrics.

In [ ]:
table(carbon$method, useNA = "always")

Calculation method:

* There are twice as many location-based values than market-based.

* Many records do not have a stated method with missing values encoded as an empty string ("") rather than NA.

*method* is conditional on *kpi*.

In [ ]:
table(carbon$kpi, carbon$method, useNA = "always")

Consider the 'reasonableness' of these statistics given how values are defined and related.

* Location-based metrics are simpler to calculate, so are the most often published.

* Market-based metrics are usually ony applied to Scope 2 calculations.

Given the above, emissions records with no method are assumed to be location-based.

In [ ]:
carbon<- carbon %>%
  mutate(method = ifelse(kpi %like% "Scope" & method == "", 'Location', method)) %>%
  as.data.frame()

table(carbon$kpi, carbon$method, useNA = "always")

Energy is reported absolute KWh (or multiple) so *method* is not applicable.

In [ ]:
carbon<- carbon %>%
  mutate(method = ifelse(carbon$kpi == "Energy" & carbon$method == "", NA, method)) %>%
  as.data.frame()

table(carbon$kpi, carbon$method, useNA = "always")

#### Business Scope

Business scope is the geographical/organisational scope of a metric.

In [ ]:
table(carbon$retailer, carbon$business_scope, useNA = "always")

* Most values relate to UK business only, but Tescos and also M&S report for their non-UK operations, which are considerably smaller than their UK operations.

* Sainsburys report metrics for both their eponymous supermarket (UK) and combined supermaket and Homebase hardware operations.

Following summation, metrics will be preferntially selected by business scope.

#### Year

In [ ]:
# How many records are there for each year

table(carbon$year, useNA = "always")

ggplot(carbon, aes(year)) +
 geom_bar() +
 xlab("Year") +
 ylab("Count") +
 theme_classic()

KPI year varies with time:

* The high number of records for 2005/6 are baselines.

* The high number of records for 2015 may result from the setting of new baselines following recalculation.

* Record numbers plateau from 2019 until 2023.

* Only 12 records for 2024 indicating values are available foe one or two retailers.

Variation in the number of metrics per year nust be taken into account when critically evaluating pattern.

# Summarising Metrics

## Multiple Reported Values

UKSmktComp contains multiple estimates for some retailer, metric, year combinations from systematic and ad-hoc recalculation.

In [ ]:
# Number of published values for Tesco Scope 1 Location-based emissions by year
carbon %>%
  filter(retailer_code == "TESC") %>%
  group_by(year) %>%
  filter(kpi == "Scope 1" & kpi_type == "Measure" & method == "Location") %>%
  summarise(n_values = n(), .groups = "drop_last") %>%
  arrange(year)

* The 2006 baseline has been recalculated five times since first calculated.

Let's explore the published values for one metric and supermarket as a scatter plot with symbol sized by value.

In [ ]:
data <- carbon %>%
  filter(retailer_code == "TESC", kpi == "Scope 1" & kpi_type == "Measure" & method == "Location")

In [ ]:
ggplot(data, aes(x = year, y = source_year, size = as.numeric(value))) +
  geom_point() +
  xlab("Value Year") +
  ylab("Source Year") +
  scale_x_continuous(limits = c(2005, 2025)) +
  scale_y_continuous(limits = c(2010, 2025)) +
  theme_classic() +
  theme(legend.position = "none")

## Calculate Mean Values

The mean of repeated values provides a single 'smoothed' estimate of retailer metrics by year, but has the disadvantage of distorting the mathematical relationships between metrics as the number and history of values vary between metrics.

In [ ]:
# Calculate the mean value of repeated retailer kpis by year
carbon.yr <- carbon %>%
   mutate(value = suppressWarnings(as.numeric(value))) %>%
   group_by(retailer_code, kpi, kpi_type, method,
      year, unit, unit_neum, unit_denom_1, unit_denom_2, business_scope) %>%
         summarize(value = mean(value, na.rm=TRUE), .groups = "drop_last")
nrow(carbon.yr)

## Select Business Scope

KPIs are reported for one or more 'business scopes' (Table 2).

<figcaption>Table 2. UK Supermarket Retailer Scopes</figcaption>

| Business Scope | Description |
|------|-------------|
| UK | UK supermarket operations only |
| UK+ | All UK operations <sup>1</sup> |
| UK and Ireland | UK and Ireland operationa |
| UK and Europe | UK and European operations |
| Global | Global operations |

<sup>1</sup> J Sainsbury plc, Sainsburys, Homebase, and Argos businesses.

* The business scope closest to UK Supermarket is preferredin the order 'UK', 'UK+', 'UK and Europe' and 'Global'.

In [ ]:
carbon.yr.best <- carbon.yr %>%
  mutate(business_scope_order = factor(business_scope,
                                       levels = c('UK','UK+','UK and Europe','Global'),
                                       ordered = TRUE)) %>%
  group_by(retailer_code, year, kpi, kpi_type, method, unit, unit_neum, unit_denom_1, unit_denom_2) %>%
  arrange(business_scope_order) %>%
  slice(1) %>%
  ungroup() %>%
  select(-business_scope_order)
nrow(carbon.yr.best)

## Add Retailer Store Statistics

Mean metric values are joined to the summarised retail points data created in workbook 2.

In [ ]:
# Download geolytix_retailpoints_v34_202412_summary.csv from GitHub
url <- "https://raw.githubusercontent.com/davidmkidd/UK-Supermarket-Carbon-Emissions/refs/heads/main/geolytix_retailpoints_v34_202412_summary.csv"
download_path <- "/content/geolytix_retailpoints_v34_202412_summary.csv"
download.file(url, destfile = download_path, mode = "wb")
store.yr <- read.csv("/content/geolytix_retailpoints_v34_202412_summary.csv", header=TRUE, stringsAsFactors=FALSE)

In [ ]:
retailer.yr <- merge(carbon.yr.best, store.yr,
   by.x = c("retailer_code", "year"),
   by.y = c("retailer_code", "year"),
   all.x = TRUE)

# Export

All data has been cleaned and processed so it is time to look at what we are really interested in – 'how absolute and standardised kpi vary with retailer_code through time?'

In [ ]:
 # Save file for next tutorial
 write.csv(retailer.yr,file='/content/retailer_emissions_yr.csv', row.names=FALSE)

---

Back: [Main Page](https://colab.research.google.com/drive/1f8a0pXfF9PqCujiwjf4TO4-k7ezt-6b3?usp=sharing)

Next: [Scope 1 Emissions](https://colab.research.google.com/drive/16AqE3DL0PjPa4ivgP8Jzap42kHXiM6FI?usp=sharing)

---